# Diffusion Drive with the Waymo Dataset

In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch_waymo import WaymoDataset

import numpy as np
from sklearn.cluster import KMeans

import cv2
import os


In [12]:
class WaymoTrajDataset(Dataset):
    def __init__(self, root, split):
        from torch_waymo import WaymoDataset
        self.ds = WaymoDataset(root, split)

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        frame = self.ds[idx]
        bev = torch.tensor(frame.bev, dtype=torch.float32)     # (C,H,W)
        traj = torch.tensor(frame.ego_future, dtype=torch.float32)  # (T,2)
        return bev, traj


In [14]:
train_dataset = WaymoTrajDataset('/data/pytorch-e2e/converted', 'training')


In [15]:
def compute_anchors(all_trajs, K=20):
    flat = all_trajs.reshape(len(all_trajs), -1)
    kmeans = KMeans(n_clusters=K).fit(flat)
    anchors = kmeans.cluster_centers_.reshape(K, -1, 2)
    return torch.tensor(anchors, dtype=torch.float32)


In [ ]:
def get_alpha_bar(t, T=1000):
    return (1 - 0.02) ** t   # simple power schedule


In [ ]:
def diffuse(anchor, t):
    a = get_alpha_bar(t)
    eps = torch.randn_like(anchor)
    noisy = torch.sqrt(a) * anchor + torch.sqrt(1 - a) * eps
    return noisy, eps


In [ ]:
def ddim_update(x, x_pred, t):
    a = get_alpha_bar(t)
    a_prev = get_alpha_bar(max(t-1, 1))
    eps = (x - torch.sqrt(a) * x_pred) / torch.sqrt(1 - a)
    x_prev = torch.sqrt(a_prev)*x_pred + torch.sqrt(1-a_prev)*eps
    return x_prev


In [ ]:
class MLP(nn.Module):
    def __init__(self, inp, out):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(inp, 256),
            nn.ReLU(),
            nn.Linear(256, out),
        )
    def forward(self, x):
        return self.net(x)


In [ ]:
class TimeEmbed(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(1, dim),
            nn.ReLU(),
            nn.Linear(dim, dim)
        )

    def forward(self, t):
        return self.fc(t.unsqueeze(-1))


In [ ]:
class CrossAttention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.q = nn.Linear(dim, dim)
        self.k = nn.Linear(dim, dim)
        self.v = nn.Linear(dim, dim)

    def forward(self, traj, bev):
        B, C, H, W = bev.shape
        bev = bev.view(B, C, H*W).permute(0,2,1)
        q = self.q(traj)
        k = self.k(bev)
        v = self.v(bev)
        att = torch.softmax((q @ k.transpose(-2,-1)) / (q.shape[-1]**0.5), dim=-1)
        return att @ v


In [ ]:
class DiffusionDecoder(nn.Module):
    def __init__(self, T, dim=128):
        super().__init__()
        self.T = T
        self.traj_enc = MLP(2, dim)
        self.bev_proj = nn.Conv2d(1, dim, 1)
        self.time_emb = TimeEmbed(dim)
        self.cross = CrossAttention(dim)
        self.ffn = MLP(dim, dim)
        self.score_head = nn.Linear(dim, 1)
        self.offset_head = nn.Linear(dim, 2)

    def forward(self, traj, bev, t):
        bev = self.bev_proj(bev)
        feat = self.traj_enc(traj)
        feat = feat + self.time_emb(t)
        feat = feat + self.cross(feat, bev)
        feat = self.ffn(feat)
        score = self.score_head(feat.mean(dim=1))
        offset = self.offset_head(feat)
        tau_hat = traj + offset
        return score, tau_hat


In [ ]:
class DiffusionDrive(nn.Module):
    def __init__(self, anchors, T=1000):
        super().__init__()
        self.anchors = anchors
        self.K = anchors.shape[0]
        self.T = T
        self.decoder = DiffusionDecoder(T)

    def forward(self, bev, traj_gt):
        B = bev.shape[0]
        t = torch.randint(1, self.T, (B,), device=bev.device)
        k = torch.randint(0, self.K, (B,), device=bev.device)

        anchor = self.anchors[k].to(bev.device)
        traj_noisy, _ = diffuse(anchor, t)

        score, tau_hat = self.decoder(traj_noisy, bev, t.float())
        return score, tau_hat, traj_noisy


In [ ]:
def diffusion_loss(tau_hat, traj_gt, score):
    Lrec = F.l1_loss(tau_hat, traj_gt)
    y = torch.ones_like(score)
    Lbce = F.binary_cross_entropy_with_logits(score, y)
    return Lrec + 0.1*Lbce


In [ ]:
def train(model, loader, optim, device):
    model.train()
    for bev, traj_gt in loader:
        bev = bev.to(device)
        traj_gt = traj_gt.to(device)

        score, tau_hat, _ = model(bev, traj_gt)
        loss = diffusion_loss(tau_hat, traj_gt, score)

        optim.zero_grad()
        loss.backward()
        optim.step()

        print("loss:", loss.item())


In [ ]:
@torch.no_grad()
def infer(model, bev, N=20):
    model.eval()
    bev = bev.unsqueeze(0)
    samples = []

    for i in range(N):
        k = torch.randint(0, model.K, (1,), device=bev.device)
        tau = model.anchors[k].to(bev.device)

        for t in reversed(range(1, model.T, 200)):  
            score, tau_hat = model.decoder(tau, bev, torch.tensor([t],device=bev.device).float())
            tau = ddim_update(tau, tau_hat, t)

        samples.append((score.item(), tau.squeeze(0).cpu()))

    samples.sort(key=lambda x: -x[0])
    return samples[0][1]


In [ ]:
# Extract all trajectories from dataset (slow, run once)
all_trajs = []
for i in range(50000):
    _, traj = train_dataset[i]
    all_trajs.append(traj.numpy())
anchors = compute_anchors(np.array(all_trajs), K=20)
torch.save(anchors, "anchors.pt")


In [ ]:
device = "cuda"

train_ds = WaymoTrajDataset("/data/waymo", "training")
train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)

anchors = torch.load("anchors.pt")
model = DiffusionDrive(anchors).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

train(model, train_loader, optimizer, device)


In [ ]:
bev, traj_gt = train_ds[0]
pred_traj = infer(model, bev.to(device))
print(pred_traj.shape)  # (T,2)
